# EUV Stochastic Printing
Ryan Sapp · CS 427 Project 3 · November 2026

## Imports

In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error

## Load data

In [2]:
rows_full = pd.read_csv("../data/full/rows.csv")
rows_test = pd.read_csv("../data/test/rows.csv")

print("full:", rows_full.shape)
print("test:", rows_test.shape)

full: (20160, 40)
test: (2696, 40)


## Inspect columns

In [3]:
for c in rows_full.columns:
    print(c)

resist_id
dose_index
seed
pitch_nm
cd_target_nm
cd_ratio
mask_cd_nm
mask_bias_nm
dose_mjcm2
d0_mjcm2
dose_factor
sigma_e_nm
sigma_a_nm
sigma_q_nm
qe
rho_pag
rho_q
rho_pg
k_amp
thickness_nm
absorptivity_um
dev_threshold
na
absorbed_frac
photons_abs_per_nm2
image_peak
nils
n_holes
n_ok
n_missing
n_merged
n_merge_events
n_neighbor_pairs
p_missing
p_merged
mean_cd_nm
mean_cd_se
sigma_cd_nm
sigma_cd_se
lcdu_3s_nm


In [4]:
print(rows_full[["na", "k_amp", "sigma_q_nm"]].nunique())

na            1
k_amp         1
sigma_q_nm    1
dtype: int64


## Select features and targets

### Feature selection

A column qualifies as a model input only if it is something that could be
set before running the process, and is not derivable from the result.
14 of the 40 columns pass; the other 26 fail one of those two tests.

| Excluded | Why |
|---|---|
| `resist_id`, `dose_index`, `seed` | Identifiers, not physical settings. `seed` is the random number generator seed|
| `cd_target_nm`, `mask_cd_nm`, `dose_mjcm2` | Already implied by features I kept. `dose_mjcm2` is `d0_mjcm2 × dose_factor`. |
| `na`, `k_amp`, `sigma_q_nm` | Identical in all 20,160 rows. A column that never varies cannot explain variation. |
| `absorbed_frac`, `photons_abs_per_nm2`, `image_peak`, `nils` | Intermediate physics the simulator computes from the settings. Including them would mean predicting the outcome from settings plus part of the answer. |
| `n_holes`, `n_ok`, `n_missing`, `n_merged`, `n_merge_events`, `n_neighbor_pairs`, `p_merged` | Simulation outputs. Using one output to predict another is leakage. |
| `mean_cd_se`, `sigma_cd_se`, `lcdu_3s_nm` | Derived from the targets themselves — `lcdu_3s_nm` is `3 × sigma_cd_nm`. |

The 14 that remain are exactly the quantities an engineer chooses before
exposing a wafer, which is what makes the trained model usable as a
substitute for running the simulator.

In [5]:
FEATURES = [
    "pitch_nm", "cd_ratio", "mask_bias_nm", "d0_mjcm2",
    "sigma_e_nm", "sigma_a_nm", "qe", "rho_pag", "rho_pg",
    "thickness_nm", "absorptivity_um", "dev_threshold",
    "rho_q",
    "dose_factor",
]

TARGETS = ["p_missing", "mean_cd_nm", "sigma_cd_nm"]

## Build film keys and check for collisions

In [6]:
rows_full["run"] = "full"
rows_test["run"] = "test"

for df in (rows_full, rows_test):
    df["resist_key"] = df["run"] + "_" + df["resist_id"].astype(str)

overlap = set(rows_full["resist_key"]) & set(rows_test["resist_key"])
print("full films:", rows_full["resist_key"].nunique())
print("test films:", rows_test["resist_key"].nunique())
print("overlapping keys:", len(overlap))

full films: 2520
test films: 337
overlapping keys: 0


## Separate training and test data sets

In [7]:
train = rows_full
test  = rows_test

print("train:", train.shape, "| test:", test.shape)

train: (20160, 42) | test: (2696, 42)


## Define targets and row filters

In [8]:
def masks_for(df):
    return {
        "p_missing":   pd.Series(True, index=df.index),
        "mean_cd_nm":  df["n_ok"] > 0,
        "sigma_cd_nm": df["n_ok"] >= 100,
    }

MASKS = masks_for(train)

for t, m in MASKS.items():
    print(f"rows for {t:14s} {m.sum()}")

print("\nrows where nothing printed:", (train["n_ok"] == 0).sum())
print("mean_cd_nm missing:", train["mean_cd_nm"].isna().sum())

rows for p_missing      20160
rows for mean_cd_nm     20114
rows for sigma_cd_nm    19909

rows where nothing printed: 46
mean_cd_nm missing: 46


## Transform inputs

Thirteen of the 14 inputs are log-transformed because they act multiplicatively
on photon or acid counts, so their effects are closer to linear in log space.
mask_bias_nm is left alone: it is an additive length offset, not a multiplier.

p_missing is modelled as log10(p + EPS). Failure rates span orders of magnitude,
and on a raw scale a model is scored almost entirely on the difference between
10% and 50% failures while treating 1-in-10,000 and 1-in-100,000 as identical.
EPS is half the smallest measurable rate, so rows with zero observed failures
read as "below detection" rather than log(0).

In [9]:
print(train[FEATURES].min()) 

pitch_nm           32.000000
cd_ratio            0.420016
mask_bias_nm        0.001276
d0_mjcm2           20.001136
sigma_e_nm          1.200149
sigma_a_nm          4.000435
qe                  2.000203
rho_pag             0.120011
rho_pg              1.500444
thickness_nm       30.010655
absorptivity_um     4.200338
dev_threshold       0.350096
rho_q               0.012274
dose_factor         0.600078
dtype: float64


In [10]:
LOG_FEATURES = [f for f in FEATURES if f != "mask_bias_nm"]
EPS = 1 / (2 * 1800)

def make_X(df):
    X = df[FEATURES].copy()
    X[LOG_FEATURES] = np.log(X[LOG_FEATURES])
    return X

def make_y(df, target):
    if target == "p_missing":
        return np.log10(df["p_missing"] + EPS)
    return df[target]

X_train = make_X(train)
print(X_train.shape, "|", X_train.isna().sum().sum(), "NaNs")

(20160, 14) | 0 NaNs


## Cross-validation setup

In [ ]:
groups = train["resist_key"]
cv = GroupKFold(n_splits=5)

MODELS = {
    "Linear": LinearRegression(),
    "Random forest": RandomForestRegressor(
        n_estimators=200, min_samples_leaf=5, random_state=0, n_jobs=-1),
    "Neural net": MLPRegressor(
        hidden_layer_sizes=(64, 64), alpha=1e-3, max_iter=1000, random_state=0),
}

for i, (tr, te) in enumerate(cv.split(X_train, groups=groups)):
    shared = set(groups.iloc[tr]) & set(groups.iloc[te])
    print(f"fold {i}: train {len(tr):6d}  test {len(te):5d}  shared films: {len(shared)}")

fold 0: train  16128  test  4032  shared films: 0
fold 1: train  16128  test  4032  shared films: 0
fold 2: train  16128  test  4032  shared films: 0
fold 3: train  16128  test  4032  shared films: 0
fold 4: train  16128  test  4032  shared films: 0


## Train and compare

In [12]:
results = []
scores = {}

for target in TARGETS:
    sub = train[MASKS[target]]
    X = make_X(sub)
    y = make_y(sub, target)
    groups_sub = sub["resist_key"]

    for name, model in MODELS.items():
        pipe = Pipeline([("scale", StandardScaler()), ("model", model)])
        yhat = cross_val_predict(pipe, X, y, groups=groups_sub, cv=cv)

        r2   = r2_score(y, yhat)
        rmse = np.sqrt(mean_squared_error(y, yhat))

        results.append({"target": target, "model": name, "r2": r2, "rmse": rmse})
        scores[(target, name)] = r2

        print(f"{target:14s} {name:14s} R2={r2:7.4f}  RMSE={rmse:.4f}")

results_df = pd.DataFrame(results)

p_missing      Linear         R2= 0.5011  RMSE=0.7429
p_missing      Random forest  R2= 0.9737  RMSE=0.1704
p_missing      Neural net     R2= 0.9864  RMSE=0.1227
mean_cd_nm     Linear         R2= 0.9809  RMSE=1.1053
mean_cd_nm     Random forest  R2= 0.9960  RMSE=0.5066
mean_cd_nm     Neural net     R2= 0.9996  RMSE=0.1693
sigma_cd_nm    Linear         R2= 0.7186  RMSE=0.1964
sigma_cd_nm    Random forest  R2= 0.9463  RMSE=0.0858
sigma_cd_nm    Neural net     R2= 0.9760  RMSE=0.0573


## Held-out test set

In [13]:
test_masks = masks_for(test)
final = []

for target in TARGETS:
    sub_tr = train[MASKS[target]]
    sub_te = test[test_masks[target]]

    X_tr, y_tr = make_X(sub_tr), make_y(sub_tr, target)
    X_te, y_te = make_X(sub_te), make_y(sub_te, target)

    for name, model in MODELS.items():
        pipe = Pipeline([("scale", StandardScaler()), ("model", model)])
        pipe.fit(X_tr, y_tr)
        yhat = pipe.predict(X_te)
        r2   = r2_score(y_te, yhat)
        rmse = np.sqrt(mean_squared_error(y_te, yhat))

        final.append({"target": target, "model": name,
                      "r2": r2, "rmse": rmse, "n_test": len(y_te)})

        print(f"{target:14s} {name:14s} R2={r2:7.4f}  n={len(y_te)}")

final_df = pd.DataFrame(final)

p_missing      Linear         R2= 0.5027  n=2696
p_missing      Random forest  R2= 0.9743  n=2696
p_missing      Neural net     R2= 0.9892  n=2696
mean_cd_nm     Linear         R2= 0.9810  n=2692
mean_cd_nm     Random forest  R2= 0.9963  n=2692
mean_cd_nm     Neural net     R2= 0.9997  n=2692
sigma_cd_nm    Linear         R2= 0.7192  n=2659
sigma_cd_nm    Random forest  R2= 0.9453  n=2659
sigma_cd_nm    Neural net     R2= 0.9815  n=2659


## Leakage measurement

In [14]:
cv_random = KFold(n_splits=5, shuffle=True, random_state=0)

leak = []
for target in TARGETS:
    sub = train[MASKS[target]]
    X = make_X(sub)
    y = make_y(sub, target)
    
    for name, model in MODELS.items():
        pipe = Pipeline([("scale", StandardScaler()), ("model", model)])
        yhat_random = cross_val_predict(pipe, X, y, cv=cv_random)

        r2_random = r2_score(y, yhat_random)
        r2_grouped = scores[(target, name)]

        leak.append({"target": target, "model": name,
                     "grouped": r2_grouped, "random": r2_random,
                     "inflation": r2_random - r2_grouped})
        print(f"{target:14s} {name:14s} grouped={r2_grouped:.4f}  random={r2_random:.4f}")

leak_df = pd.DataFrame(leak)

p_missing      Linear         grouped=0.5011  random=0.5004
p_missing      Random forest  grouped=0.9737  random=0.9740
p_missing      Neural net     grouped=0.9864  random=0.9869
mean_cd_nm     Linear         grouped=0.9809  random=0.9809
mean_cd_nm     Random forest  grouped=0.9960  random=0.9962
mean_cd_nm     Neural net     grouped=0.9996  random=0.9997
sigma_cd_nm    Linear         grouped=0.7186  random=0.7186
sigma_cd_nm    Random forest  grouped=0.9463  random=0.9501
sigma_cd_nm    Neural net     grouped=0.9760  random=0.9803


## Save Results

In [15]:
os.makedirs("../results/ml", exist_ok=True)

results_df.to_csv("../results/ml/cv_scores.csv", index=False)
leak_df.to_csv("../results/ml/leakage_comparison.csv", index=False)
final_df.to_csv("../results/ml/heldout_scores.csv", index=False)

print("saved to results/ml/")

saved to results/ml/
